In [17]:
import sys; sys.path.append('..')
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX
import joblib
from src.model import score_params, train_quantile_models
from src.backtest import expanding_window_splits
from src.metrics import mase, seasonal_naive_scale, seasonal_naive_pred

model_df = pd.read_pickle('../data/model_df.pkl')

ImportError: cannot import name 'score_params' from 'src.model' (/Users/jay/Documents/Project 2/demand_forecast_inventory_optimizer/notebooks/../src/model.py)

In [ ]:
# Verify lag_7 really equals sales from 7 days earlier
# rolling mean uses only prior days

one = model_df[model_df['id'] == model_df['id'].iloc[0]].head(20)
one[['date', 'sales', 'lag_7', 'rolling_mean_7', 'rolling_std_7']]

,date,sales,lag_7,rolling_mean_7,rolling_std_7
0,2011-02-26,2,1.0,1.857143,1.214986
1,2011-02-27,2,2.0,2.000000,1.154701
2,2011-02-28,0,0.0,2.000000,1.154701
3,2011-03-01,2,2.0,2.000000,1.154701
4,2011-03-02,1,2.0,2.000000,1.154701
5,2011-03-03,7,2.0,1.857143,1.214986
6,2011-03-04,1,4.0,2.571429,2.299068
7,2011-03-05,2,2.0,2.142857,2.267787
8,2011-03-06,3,2.0,2.142857,2.267787
9,2011-03-07,0,0.0,2.285714,2.288689


In [ ]:
feature_cols = (
    [f'lag_{l}' for l in (7, 14, 28)]
    + [f'rolling_mean_{w}' for w in (7, 28)]
    + [f'rolling_std_{w}' for w in (7, 28)]
    + ['wday', 'month', 'is_event', 'snap', 'day', 'sell_price']
)

before = len(model_df)

#Drop rows with NaN features (first 28 days of each series, plus any missing prices)
model_df = model_df.dropna(subset=feature_cols).reset_index(drop=True)
print(f"Dropped {before - len(model_df)} rows with NaN features; {len(model_df)} remain.")
print(f"Feature columns: {feature_cols}")

Dropped 0 rows with NaN features; 2241684 remain.
Feature columns: ['lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28', 'wday', 'month', 'is_event', 'snap', 'day', 'sell_price']


In [ ]:
for i, (train_idx, test_idx) in enumerate(expanding_window_splits(model_df)):
    tr, te = model_df.loc[train_idx], model_df.loc[test_idx]
    print(f"Fold {i}:")
    print(f"  train {tr['date'].min().date()} → {tr['date'].max().date()}  ({len(tr):,} rows)")
    print(f"  test  {te['date'].min().date()} → {te['date'].max().date()}  ({len(te):,} rows)")
    print()

Fold 0:
  train 2011-02-26 → 2016-04-24  (2,201,448 rows)
  test  2016-04-25 → 2016-05-22  (40,236 rows)

Fold 1:
  train 2011-02-26 → 2016-03-27  (2,161,212 rows)
  test  2016-03-28 → 2016-04-24  (40,236 rows)

Fold 2:
  train 2011-02-26 → 2016-02-28  (2,120,976 rows)
  test  2016-02-29 → 2016-03-27  (40,236 rows)



In [ ]:
for i, (train_idx, test_idx) in enumerate(expanding_window_splits(model_df)):
    tr, te = model_df.loc[train_idx], model_df.loc[test_idx]
    scale = seasonal_naive_scale(tr)
    score = mase(te['sales'].values, seasonal_naive_pred(te).values, scale)
    print(f"Fold {i}: seasonal-naive MASE = {score:.3f}")

Fold 0: seasonal-naive MASE = 0.948
Fold 1: seasonal-naive MASE = 0.907
Fold 2: seasonal-naive MASE = 0.868


In [ ]:
import warnings 
from statsmodels.tsa.statespace.sarimax import SARIMAX
warnings.filterwarnings("ignore")  # statsmodels is noisy

# Picking the 3 highest-volume series to keep the example small and fast. You can try more series, but it will take longer.
top_ids = (model_df.groupby('id')['sales'].sum()
           .sort_values(ascending=False).head(3).index.tolist())

Horizon = 28
results = []

for sid in top_ids:
    s = model_df[model_df['id'] == sid].sort_values('date')
    y = s['sales'].values
    train, test = y[:-Horizon], y[-Horizon:]

    # simple weekly-seasonal spec
    model = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,7),
                    enforce_stationarity=False, enforce_invertibility=False)
    fit = model.fit(disp=False)
    fcst = fit.forecast(steps=Horizon)

    #MASE using the per-series seasonal-naive scale on this series
    scale = np.mean(np.abs(train[7:] - train[:-7]))
    score = np.mean(np.abs(test - fcst)) / scale if scale != 0 else np.nan
    results.append((sid, score))
    print(f"{sid}: SARIMA MASE = {score: .3f}")

print("Per-series comparison (same series, same scaling):")
for sid in top_ids:
    s = model_df[model_df['id'] == sid].sort_values('date')
    y = s['sales'].values
    train, test = y[:-Horizon], y[-Horizon:]
    scale = np.mean(np.abs(train[7:] - train[:-7]))

    # seasonal-naive forecast for the 28-day test window = value 7 days earlier
    naive_fcst = y[-Horizon-7:-7]
    naive_mase = np.mean(np.abs(test - naive_fcst)) / scale
    print(f"{sid}: seasonal-naive MASE = {naive_mase:.3f}")

print("\nMean SARIMA MASE:", np.nanmean([r[1] for r in results]))   

FOODS_3_090_CA_1_evaluation: SARIMA MASE =  0.428
FOODS_3_586_CA_1_evaluation: SARIMA MASE =  0.606
FOODS_3_252_CA_1_evaluation: SARIMA MASE =  0.603
Per-series comparison (same series, same scaling):
FOODS_3_090_CA_1_evaluation: seasonal-naive MASE = 0.510
FOODS_3_586_CA_1_evaluation: seasonal-naive MASE = 0.716
FOODS_3_252_CA_1_evaluation: seasonal-naive MASE = 0.721

Mean SARIMA MASE: 0.545717394660517


In [ ]:
# --- Feature setup ---
# Categorical features LightGBM should treat as such
cat_features = ['dept_id', 'wday', 'month']
# Make sure categoricals are 'category' dtype
for c in cat_features:
    model_df[c] = model_df[c].astype('category')

# Full feature list: your engineered features + categoricals
features = (
    ['lag_7', 'lag_14', 'lag_28',
     'rolling_mean_7', 'rolling_std_7', 'rolling_mean_28', 'rolling_std_28',
     'day', 'is_event', 'snap', 'sell_price']
    + cat_features
)
target = 'sales'

In [ ]:
def run_backtest(model_name):
    """Train one model type across the expanding-window folds, return MASE per fold."""
    scores = []
    for i, (train_idx, test_idx) in enumerate(expanding_window_splits(model_df)):
        tr, te = model_df.loc[train_idx], model_df.loc[test_idx]
        X_tr, y_tr = tr[features], tr[target]
        X_te, y_te = te[features], te[target]

        if model_name == 'lightgbm':
            model = lgb.LGBMRegressor(
                n_estimators=300, learning_rate=0.05,
                num_leaves=63, n_jobs=-1, verbose=-1
            )
            model.fit(X_tr, y_tr, categorical_feature=cat_features)
        else:  # xgboost
            model = xgb.XGBRegressor(
                n_estimators=300, learning_rate=0.05,
                max_depth=6, n_jobs=-1, enable_categorical=True
            )
            model.fit(X_tr, y_tr)

        pred = model.predict(X_te)
        pred = np.clip(pred, 0, None)  # sales can't be negative

        scale = seasonal_naive_scale(tr)        # same scale as the baseline
        score = np.mean(np.abs(y_te.values - pred)) / scale
        scores.append(score)
        print(f"  Fold {i}: MASE = {score:.3f}")
    print(f"  Mean MASE = {np.mean(scores):.3f}\n")
    return scores

print("LightGBM:")
lgbm_scores = run_backtest('lightgbm')

print("XGBoost:")
xgb_scores  = run_backtest('xgboost')

LightGBM:
  Fold 0: MASE = 0.742
  Fold 1: MASE = 0.709
  Fold 2: MASE = 0.694
  Mean MASE = 0.715

XGBoost:
  Fold 0: MASE = 0.745
  Fold 1: MASE = 0.710
  Fold 2: MASE = 0.695
  Mean MASE = 0.716



In [ ]:
print(f"{'Model':<22}{'Mean MASE':>10}")
print("-" * 32)
print(f"{'Seasonal-naive (all)':<22}{0.908:>10.3f}")   
print(f"{'LightGBM (global)':<22}{np.mean(lgbm_scores):>10.3f}")
print(f"{'XGBoost (global)':<22}{np.mean(xgb_scores):>10.3f}")

Model                  Mean MASE
--------------------------------
Seasonal-naive (all)       0.908
LightGBM (global)          0.715
XGBoost (global)           0.716


In [ ]:
import lightgbm as lgb
import numpy as np
import joblib
from itertools import product

# Reuse the feature setup from your comparison cell
cat_features = ['dept_id', 'wday', 'month']
for c in cat_features:
    model_df[c] = model_df[c].astype('category')

features = (
    ['lag_7', 'lag_14', 'lag_28',
     'rolling_mean_7', 'rolling_std_7', 'rolling_mean_28', 'rolling_std_28',
     'day', 'is_event', 'snap', 'sell_price'] + cat_features
)
target = 'sales'

def score_params(params):
    """Mean MASE across the time-aware folds for one param set."""
    fold_scores = []
    for train_idx, test_idx in expanding_window_splits(model_df):
        tr, te = model_df.loc[train_idx], model_df.loc[test_idx]
        model = lgb.LGBMRegressor(**params, n_jobs=-1, verbose=-1)
        model.fit(tr[features], tr[target], categorical_feature=cat_features)
        pred = np.clip(model.predict(te[features]), 0, None)
        scale = seasonal_naive_scale(tr)
        fold_scores.append(np.mean(np.abs(te[target].values - pred)) / scale)
    return np.mean(fold_scores)

# A small, sensible search space — light, not exhaustive
candidates = [
    {'n_estimators': 300, 'learning_rate': 0.05, 'num_leaves': 31},
    {'n_estimators': 500, 'learning_rate': 0.05, 'num_leaves': 63},
    {'n_estimators': 500, 'learning_rate': 0.03, 'num_leaves': 63},
    {'n_estimators': 800, 'learning_rate': 0.03, 'num_leaves': 127},
]

results = [(p, score_params(p)) for p in candidates]
for p, s in sorted(results, key=lambda x: x[1]):
    print(f"MASE {s:.3f}  ←  {p}")

best_params = min(results, key=lambda x: x[1])[0]
print("\nBest:", best_params)

MASE 0.713  ←  {'n_estimators': 800, 'learning_rate': 0.03, 'num_leaves': 127}
MASE 0.714  ←  {'n_estimators': 500, 'learning_rate': 0.05, 'num_leaves': 63}
MASE 0.716  ←  {'n_estimators': 500, 'learning_rate': 0.03, 'num_leaves': 63}
MASE 0.718  ←  {'n_estimators': 300, 'learning_rate': 0.05, 'num_leaves': 31}

Best: {'n_estimators': 800, 'learning_rate': 0.03, 'num_leaves': 127}


In [ ]:
final_model = lgb.LGBMRegressor(**best_params, n_jobs = -1, verbose =-1)
final_model.fit(model_df[features], model_df[target], categorical_feature= cat_features)

joblib.dump(final_model, '../models/lgbm_final.joblib')
print("Saved to models/lgbm_final.joblib")

Saved to models/lgbm_final.joblib


In [ ]:
# Train on the first fold training data
train_idx, test_idx = next(expanding_window_splits(model_df))
tr, te = model_df.loc[train_idx], model_df.loc[test_idx]

qmodels = train_quantile_models(tr, features, target, cat_features)

# Predict all three quartiles on the test set
preds = {q: np.clip(m.predict(te[features]), 0, None) for q, m in qmodels.items()}

# Look at a few rows: low / median / high

inspect = pd.DataFrame({
    'actual': te[target].values,
    'q10': preds[0.1],
    'q50': preds[0.5],
    'q90': preds[0.9],
}).head(15)
print(inspect)

    actual  q10       q50       q90
0        2  0.0  1.000000  2.600878
1        0  0.0  1.000000  2.369015
2        0  0.0  1.000000  2.366902
3        0  0.0  1.000000  2.441752
4        0  0.0  1.000000  2.894707
5        1  0.0  0.999413  2.450508
6        1  0.0  0.997800  2.485197
7        0  0.0  0.950847  2.612607
8        6  0.0  0.128219  2.135830
9        1  0.0  1.000000  2.591644
10       0  0.0  0.906412  2.090807
11       3  0.0  0.989263  2.631301
12       0  0.0  1.000000  3.359584
13       0  0.0  1.000000  2.661523
14       0  0.0  0.837902  2.556688
